# Zero-shot baseline: Qwen3-1.7B as a citation judge

This notebook produces the fourth pre-registered baseline: the untrained Qwen3-1.7B, asked the same question as every other judge. It exists so the fine-tune's contribution can be isolated. Whatever the adapter adds is the gap between this model and the fine-tuned one, and nothing else.

**Run it before any gold label exists.** That ordering is a hard rule, not a preference. The prediction timestamps are the audit trail that shows the baseline was fixed before anyone could see the answers. The protocol (same prompt, thinking off, 4-bit NF4, one-pass scoring) is in the [addendum to PREREGISTRATION.md](https://github.com/SheinRG/ResearchAgent/blob/main/finetune/PREREGISTRATION.md#addendum-2026-11-19-the-zero-shot-protocol).

**Runtime:** Google Colab with a T4 GPU (Runtime > Change runtime type > T4 GPU).

**You need:** your local `backend/evals/data/pairs.jsonl`. It is gitignored because it holds third-party page text, so it is not in the clone and you will upload it.

Steps:

1. Setup and the gold set
2. Load the model
3. One judgement
4. The full run and export

## 1. Setup

Check the GPU, install the dependencies, and pull the repo so the prompt and the gold manifest come from the source of truth.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU found. Switch the runtime: Runtime > Change runtime type > T4 GPU, then rerun."
)
props = torch.cuda.get_device_properties(0)
print("device:", props.name)
print(f"memory: {props.total_memory / 1024**3:.1f} GB")
print("bf16 supported:", torch.cuda.is_bf16_supported())

In [ ]:
# Qwen3 needs transformers 4.51+; bitsandbytes is for the 4-bit load in step 2.
%pip install -q "transformers>=4.51" "accelerate>=0.33" "bitsandbytes>=0.43"

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/SheinRG/ResearchAgent.git"
REPO = "/content/ResearchAgent"

# Clone once; rerunning the cell must not fail or silently move the checkout.
if not os.path.exists(REPO):
    !git clone -q --depth 1 {REPO_URL} {REPO}

# Recorded so the predictions can be tied to the exact prompt/manifest used.
REPO_SHA = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()

sys.path.insert(0, f"{REPO}/backend")

# Importing the prompt rather than copying it is what guarantees zero-shot is
# asked exactly what the prompted judges were asked.
from evals.judge import _SYSTEM, build_prompt, _record, load_predictions, append_prediction, _pred_path
from evals.contamination import fingerprint, load_manifest

print("repo commit:", REPO_SHA)

## The gold set

`pairs.jsonl` is not in git because it holds third-party page text, so upload your local copy. The 200 gold pairs are defined by the committed manifest, not re-derived here. Every pair is checked against the manifest's hashes, so a stale or different capture file fails loudly instead of producing a baseline over the wrong data.

In [ ]:
import json

PAIRS_PATH = "/content/pairs.jsonl"

if not os.path.exists(PAIRS_PATH):
    from google.colab import files
    uploaded = files.upload()
    # Whatever the file was called locally, take the single upload.
    assert len(uploaded) == 1, f"expected exactly one file, got {list(uploaded)}"
    (name,) = uploaded
    os.replace(name, PAIRS_PATH)

with open(PAIRS_PATH, encoding="utf-8") as f:
    pairs = [json.loads(line) for line in f if line.strip()]
print("candidate pairs loaded:", len(pairs))

In [ ]:
from collections import Counter

manifest = load_manifest(f"{REPO}/finetune/gold_manifest.json")

by_id = {p["pair_id"]: p for p in pairs}
missing = [pid for pid in manifest["pair_ids"] if pid not in by_id]
if missing:
    raise RuntimeError(
        f"{len(missing)} gold pair ids from the manifest are not in pairs.jsonl; "
        "this is not the capture the manifest was built from."
    )

# Manifest order, so the run order is reproducible.
gold = [by_id[pid] for pid in manifest["pair_ids"]]

# Ids alone are not enough: the same id could carry different text in a
# re-captured file. Hashes pin the actual content.
evidence_ok = set(manifest["evidence_sha"])
sentence_ok = set(manifest["sentence_sha"])
mismatched = [
    p["pair_id"]
    for p in gold
    if fingerprint(p["evidence"]) not in evidence_ok
    or fingerprint(p["sentence"]) not in sentence_ok
]
if mismatched:
    raise RuntimeError(
        f"{len(mismatched)} gold pairs do not match the manifest hashes: pairs.jsonl "
        "differs from the capture the manifest was built from."
    )
assert len(gold) == manifest["n"], (len(gold), manifest["n"])

# Deliberately no labels here: none exist, and this notebook must not see any.
print("gold pairs:", len(gold))
print("pair_type:", dict(Counter(p["pair_type"] for p in gold)))
print("mean evidence length (words):", round(sum(len(p["evidence"].split()) for p in gold) / len(gold), 1))

## 2. Load the model

The model is loaded in 4-bit NF4 because that is what the QLoRA adapter will run on, so the baseline and the fine-tune differ only by the adapter.

The revision hash is recorded because `Qwen/Qwen3-1.7B` on the Hub is a moving name: the repo can be updated, and a baseline is only meaningful against the exact weights it was run on.

In [ ]:
from huggingface_hub import model_info
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-1.7B"
# judge.py's scorer finds the zero-shot baseline by "qwen" and "zero" in the
# prediction file name, which _pred_path(JUDGE_NAME) derives.
JUDGE_NAME = "Qwen/Qwen3-1.7B zero-shot"

# The T4 has no native bf16, so compute falls back to fp16 there.
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

# The standard QLoRA recipe. The Milestone 3 training notebook must use this
# same config, or the baseline and the fine-tune differ by more than the adapter.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

# Resolve the revision once and load both pieces at it, so the tokenizer and
# weights cannot come from different commits.
MODEL_REVISION = model_info(MODEL_ID).sha
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()

print("model revision:", MODEL_REVISION)
print(f"memory footprint: {model.get_memory_footprint() / 1024**3:.2f} GB")

### What the model actually sees

The chat template wraps the prompt in Qwen's special tokens. `enable_thinking=False` makes the template emit an empty think block, so the very next token the model produces is the answer. Printing the rendered prompt once is the cheapest way to catch a template surprise.

In [ ]:
def render(pair) -> str:
    messages = [
        {"role": "system", "content": _SYSTEM},
        {"role": "user", "content": build_prompt(pair["sentence"], pair["evidence"])},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )

example = render(gold[0])
print(example)
print("-" * 40)
print("tokens in this prompt:", len(tokenizer(example)["input_ids"]))

# A long chunk silently truncated would judge a different pair than the one
# asked about; the max over the whole gold set shows there is no such risk.
max_tokens = max(len(tokenizer(render(p))["input_ids"]) for p in gold)
print("max tokens over gold:", max_tokens)

assert "<think>\n\n</think>\n\n" in example[-40:], (
    "The chat template no longer ends with an empty think block: it changed, "
    "and thinking may be on. Do not run the baseline until this is understood."
)

### The two answer words, as tokens

The model does not see words, it sees tokens. We read the probability of the *first* token of each answer, so the two answers must start with different tokens. We also sum over the spellings the model might plausibly use (case, leading space) so a capital letter does not hide probability mass.

In [ ]:
VARIANTS = {
    "supported": ["supported", "Supported", " supported", " Supported"],
    "unsupported": ["unsupported", "Unsupported", " unsupported", " Unsupported"],
}

ANSWER_IDS = {}
for label, variants in VARIANTS.items():
    first_ids = set()
    for v in variants:
        ids = tokenizer.encode(v, add_special_tokens=False)
        print(f"{label:12s} {v!r:16} -> {tokenizer.convert_ids_to_tokens(ids)}")
        first_ids.add(ids[0])
    ANSWER_IDS[label] = sorted(first_ids)

# The important check. If any variant of one answer starts with a token that
# also starts a variant of the other, the first-token probability cannot tell
# the two classes apart and the score would be meaningless.
overlap = set(ANSWER_IDS["supported"]) & set(ANSWER_IDS["unsupported"])
assert not overlap, (
    f"'supported' and 'unsupported' share first tokens {sorted(overlap)} "
    f"({tokenizer.convert_ids_to_tokens(sorted(overlap))}); first-token scoring "
    "cannot separate them."
)

print("ANSWER_IDS:", ANSWER_IDS)